In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping

import joblib

In [1]:
# Load the requested data sets.
infra_metrics_df = pd.read_csv('/Users/raj/Desktop/CapStone/bankOps-platform/DataSets/infrastructure_metrics.csv')


NameError: name 'pd' is not defined

In [2]:
infra_metrics_df['timestamp'] = pd.to_datetime(infra_metrics_df['timestamp'])

infra_metrics_df = infra_metrics_df.sort_values('timestamp').reset_index(drop=True)

NameError: name 'pd' is not defined

In [ ]:
infra_metrics_df.head()

In [ ]:
features = [
    'cpu_utilization',
    'memory_usage',
    'db_connection_pool_usage',
    'network_throughput_mbps',
    'application_error_rate',
    'neft_success_rate',
    'imps_success_rate',
    'rtgs_success_rate',
    'avg_transaction_processing_time_ms',
    'auth_success_rate',
    'transaction_volume_per_hour',
    'service_uptime'
]

X = infra_metrics_df[features].copy()

In [ ]:
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
sequence_length = 60

In [ ]:
def create_sequences(data, sequence_length):

    sequences = []

    for i in range(len(data) - sequence_length + 1):

        sequence = data[i:i + sequence_length]

        sequences.append(sequence)

    return np.array(sequences)

In [ ]:
X_train_seq = create_sequences(
    X_train_scaled,
    sequence_length
)

X_test_seq = create_sequences(
    X_test_scaled,
    sequence_length
)

In [ ]:
print("Training sequences:", X_train_seq.shape)
print("Testing sequences:", X_test_seq.shape)

In [ ]:
n_features = X_train_seq.shape[2]

input_layer = Input(shape=(sequence_length, n_features))

# Encoder
encoded = LSTM(64, activation='tanh')(input_layer)

# Bottleneck
decoded = RepeatVector(sequence_length)(encoded)

# Decoder
decoded = LSTM(64, activation='tanh', return_sequences=True)(decoded)

# Reconstruct original features
output_layer = TimeDistributed(Dense(n_features))(decoded)

# Build model
autoencoder = Model(inputs=input_layer, outputs=output_layer)

autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

autoencoder.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train_seq,
    X_train_seq,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    shuffle=False
)

In [ ]:
autoencoder.fit(X_train_seq, X_train_seq)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('LSTM Autoencoder Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.show()

In [ ]:
# Create model artifact
rf_artifact = {
    'model': autoencoder,
    'features': features
}

# Save model artifact
joblib.dump(rf_artifact, 'anamoly_detection.pkl')

print("Anamoly Detection Model Saved successfully.")
